In [1]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [3]:
data = pd.read_csv("../data/raw/CEAS_08.csv")

print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (39154, 7)


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


In [4]:
data = data[
    ["sender", "receiver", "date", "subject", "body", "urls", "label"]
].copy()

data.head()

,sender,receiver,date,subject,body,urls,label
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,1,0
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


In [5]:
# handle missing values
data["subject"] = data["subject"].fillna("")
data["body"] = data["body"].fillna("")
data["sender"] = data["sender"].fillna("")
data["receiver"] = data["receiver"].fillna("")
data["urls"] = data["urls"].fillna(0)

In [6]:
data.isnull().sum()

sender      0
receiver    0
date        0
subject     0
body        0
urls        0
label       0
dtype: int64

In [7]:
# Combine Subject + Body - This is important because both subject and body contain phishing clues.
data["text"] = data["subject"] + " " + data["body"]

data[["subject", "body", "text", "label"]].head()

,subject,body,text,label
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...","Never agree to be a loser Buck up, your troubl...",1
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,Befriend Jenna Jameson \nUpgrade your sex and ...,1
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,1
3,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,Re: svn commit: r619753 - in /spamassassin/tru...,0
4,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,SpecialPricesPharmMoreinfo \nWelcomeFastShippi...,1


In [8]:
# Create cleaning function
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    
    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)
    
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]
    
    return " ".join(tokens)

In [9]:
# apply cleaning
data["clean_text"] = data["text"].apply(clean_text)

data[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,"Never agree to be a loser Buck up, your troubl...",never agree loser buck troubles caused small d...,1
1,Befriend Jenna Jameson \nUpgrade your sex and ...,befriend jenna jameson upgrade sex pleasures t...,1
2,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,cnncom daily top daily top cnncom top videos s...,1
3,Re: svn commit: r619753 - in /spamassassin/tru...,svn commit r spamassassintrunk libmailspamassa...,0
4,SpecialPricesPharmMoreinfo \nWelcomeFastShippi...,specialpricespharmmoreinfo welcomefastshipping...,1


In [10]:
# Check empty texts
empty_texts = (data["clean_text"].str.strip() == "").sum()

print("Empty cleaned texts:", empty_texts)

Empty cleaned texts: 0


In [11]:
data = data[data["clean_text"].str.strip() != ""].copy()

print("New Shape:", data.shape)

New Shape: (39154, 9)


In [12]:
# Save processed dataset
processed_data = data[
    ["sender", "receiver", "date", "urls", "subject", "body", "text", "clean_text", "label"]
].copy()

processed_data.to_csv(
    "../data/processed/cleaned_phishing_emails.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
